# 02 · Silver · `unidades_consumidoras`
### V-Energy Grid 360 — Arquitetura Medalhão

Lê `v_energy.bronze.unidades_consumidoras` (`unidades_consumidoras.csv`, tudo como texto) e grava `v_energy.silver.unidades_consumidoras`. **Grão: 1 linha = 1 UC.** É a tabela central do case: todas as outras se ligam a ela por `id_uc`, e a Gold monta a `dim_uc` a partir daqui.

**Fonte das regras:** `exploracao_unidades_consumidoras.ipynb` e a seção `unidades_consumidoras` do `docs/data_quality_log.md` (U1 a U10).

| # | Tratamento | Linhas afetadas | Regra | Log |
|---|---|---|---|---|
| 1 | Cabeçalho repetido como dado | 1 | remover antes de tudo | U1 |
| 2 | Mesma UC repetida com grafias diferentes de nome e documento | 1.260 | padronizar e depois `dropDuplicates(["id_uc"])` | U2 |
| 3 | UCs de teste (sem medidor, sem leitura, sem fatura) | 200 | remover | U3 |
| 4 | `uf` com `PE`, `P.E.` e `Pernambuco` | 6.990 fora do padrão | `PE` | U4 |
| 5 | Grafias de `classe`, `municipio`, `status` e espaços em textos | ver células | mapa para valores canônicos + `trim` | U5 |
| 6 | `bairro` com acentuação corrompida (mojibake) | 356 (388 suspeitas − 32 de `Rio Âmbar`, correto) | reverter a codificação + mapa manual | U6 |
| 7 | `cep` inválido | 1.383 | nulo | U7 |
| 8 | `cpf_cnpj` com máscara | ver célula | só dígitos + `tipo_pessoa` (PF/PJ) | U8 |
| 9 | `data_ligacao` impossível (antes de 1950) | 90 | data de instalação do 1º medidor + flag `data_ligacao_estimada` | U9 |
| 10 | Dados pessoais | todas | mantidos na Silver; **mascarados na Gold** | U10 |

**Ordem das operações (importa):**
1. Remover o cabeçalho antes de qualquer contagem.
2. Padronizar nome e documento **antes** de deduplicar: as repetições só diferem nessas grafias (exploração, seção 4).
3. Corrigir o mojibake **antes** do `trim` e de qualquer `upper`, porque a correção depende dos bytes originais.
4. A data de ligação estimada usa `silver.medidores`, que precisa existir (rodar `02_silver_medidores` antes).

**Resultado esperado:** 45.000 UCs (o número do case), com `id_uc` único e exatamente 1 medidor ativo por UC.

**Pré-requisito:** `silver.medidores` gravada. `silver.leituras` é usada só numa checagem (pulada se não existir).

## 1. Parâmetros do ambiente

In [0]:
catalog = "v_energy"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"
landing_path = f"/Volumes/{catalog}/bronze/landing"

from pyspark.sql import functions as F, Row
from datetime import datetime

tabela_bronze = f"{bronze_schema}.unidades_consumidoras"
tabela_silver = f"{silver_schema}.unidades_consumidoras"

def n_br(n):
    return f"{n:,}".replace(",", ".")

def sem_acento(col):
    """upper + trim + sem acento + espaços internos simples (mesma função da exploração)."""
    c = F.upper(F.trim(col))
    c = F.translate(c, "ÁÀÂÃÉÊÍÓÔÕÚÜÇ", "AAAAEEIOOOUUC")
    return F.regexp_replace(c, r"\s+", " ")

def espacos(col):
    """trim + espaços internos simples, sem mudar maiúsculas nem acentos."""
    return F.regexp_replace(F.trim(col), r"\s+", " ")

print(f"origem:  {tabela_bronze}")
print(f"destino: {tabela_silver}")

## 2. Funções de checagem de qualidade

Mesmas funções das outras Silvers (`02_silver_leituras`, `02_silver_medidores`), gravando em `silver.dq_log` com as mesmas colunas. As funções só acumulam resultados em `dq_results`; o notebook grava o log **sempre** e só grava a Silver se **todas** passarem.

In [0]:
dq_results = []

def dq_check(table_name: str, check_name: str, df, condition):
    """Executa uma checagem de qualidade: conta quantas linhas violam a condição esperada."""
    total = df.count()
    failed = df.filter(~condition).count()
    passed = failed == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=failed, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {failed}/{total} linhas falharam")

def dq_check_unique(table_name: str, check_name: str, df, key_cols: list):
    """Checagem de qualidade específica para unicidade de chave."""
    total = df.count()
    dupes = df.groupBy(*key_cols).count().filter("count > 1").count()
    passed = dupes == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=dupes, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {dupes} chaves duplicadas de {total} linhas")

def dq_check_contagem(table_name: str, check_name: str, df, esperado: int):
    """Checagem de contagem total: passa só se o número de linhas for exatamente o esperado."""
    total = df.count()
    diferenca = abs(total - esperado)
    passed = diferenca == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=diferenca, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {n_br(total)} linhas (esperado {n_br(esperado)})")

## 3. Leitura da Bronze

Leitura sem transformação. Conferência: **46.461** linhas, todas as colunas `string`. O DataFrame `bronze` fica guardado para a comparação Bronze × Silver no fim.

In [0]:
bronze = spark.table(tabela_bronze)

n_bronze = bronze.count()
print(f"Linhas na Bronze: {n_br(n_bronze)}")  # esperado: 46.461

bronze.printSchema()
display(bronze.limit(5))

## 4. Cabeçalho repetido (log U1)

**Decisão:** remover a linha em que `id_uc = 'id_uc'`.

**Evidência:** o arquivo de origem tem o cabeçalho repetido no meio dos dados (linha 27.536 do CSV). É um nulo estrutural, sem nenhuma informação. Mesma convenção de `faturas` (F1).

**Conferência:** 1 linha removida; restam 46.460.

In [0]:
# Tratamento 1 · cabeçalho repetido (esperado: 1 removida, 46.460 restantes)
df_sem_cab = bronze.filter(~F.coalesce(F.col("id_uc") == "id_uc", F.lit(False)))
n_sem_cab = df_sem_cab.count()
print(f"Cabeçalhos removidos: {n_bronze - n_sem_cab} | antes: {n_br(n_bronze)} | depois: {n_br(n_sem_cab)}")

## 5. Documento e nome padronizados, depois deduplicação (log U2 e U8)

**Decisão:**
- `cpf_cnpj`: manter **só os dígitos** (tira `.`, `-` e `/`) e criar `tipo_pessoa` = `PF` (11 dígitos) ou `PJ` (14 dígitos).
- `nome_titular`: `upper`, sem acento e com espaços simples.
- Depois disso, `dropDuplicates(["id_uc"])`.

**Evidência:** 46.460 linhas para 45.200 `id_uc` distintos. As repetições não são cópias exatas: a mesma UC aparece com o nome e o documento escritos de jeitos diferentes (com e sem máscara, com e sem acento, maiúsculas). Depois de padronizar essas duas colunas, **nenhum id continua com versões diferentes** (exploração, seção 4). Então qualquer uma das versões serve, e nenhuma informação se perde.

**Por que o nome fica sem acento:** é a forma que torna as versões iguais. O nome não é usado em métricas e será mascarado na Gold (U10).

**Conferência:** antes do `dropDuplicates`, linhas distintas = ids distintos = 45.200; depois, 45.200 linhas.

In [0]:
# Tratamento 2 · padroniza documento e nome, e só então deduplica
df_pad = (
    df_sem_cab
    .withColumn("cpf_cnpj", F.regexp_replace("cpf_cnpj", "[^0-9]", ""))
    .withColumn("tipo_pessoa", F.when(F.length("cpf_cnpj") == 11, F.lit("PF"))
                               .when(F.length("cpf_cnpj") == 14, F.lit("PJ")))
    .withColumn("nome_titular", sem_acento(F.col("nome_titular")))
)

cols_origem = [c for c in bronze.columns if c not in ("source_file", "ingestion_timestamp")]
n_ids = df_pad.select("id_uc").distinct().count()
n_versoes = df_pad.select(cols_origem).distinct().count()
# esperado: 45.200 e 45.200 (se forem diferentes, ainda há versões que a padronização não unificou)
print(f"id_uc distintos: {n_br(n_ids)} | linhas distintas depois de padronizar: {n_br(n_versoes)}")

df_dedup = df_pad.dropDuplicates(["id_uc"])
n_dedup = df_dedup.count()
print(f"Repetições removidas: {n_br(n_sem_cab - n_dedup)} | antes: {n_br(n_sem_cab)} | depois: {n_br(n_dedup)}")

# esperado: só 11 (PF) e 14 (PJ)
display(df_dedup.groupBy(F.length("cpf_cnpj").alias("qtd_digitos"), "tipo_pessoa").count().orderBy("qtd_digitos"))

## 6. UCs de teste (log U3)

**Decisão:** remover as 200 UCs que **não têm nenhum medidor** em `silver.medidores`.

**Evidência (exploração, seção 10):** são 200 UCs com status `ATIVA`, mas sem medidor, sem nenhuma leitura e, portanto, sem fatura. Os dados delas mostram que são **registros de teste** do sistema de origem, e não clientes reais. Sem elas, o total fica em **45.000, exatamente o número de UCs do case**, e cada UC restante tem exatamente 1 medidor ativo.

**Por que remover (e não manter com flag):** registro de teste não é cliente. Mantido, ele inflaria a base de clientes e o denominador do DEC/FEC. A Bronze preserva as 200 linhas, e a lista fica visível abaixo.

**Conferência:** 200 removidas; restam 45.000.

In [0]:
# Tratamento 3 · remove as UCs de teste (sem nenhum medidor)
ucs_com_medidor = spark.table(f"{silver_schema}.medidores").select("id_uc").distinct()

df_teste = df_dedup.join(ucs_com_medidor, "id_uc", "left_anti")
df_reais = df_dedup.join(ucs_com_medidor, "id_uc", "left_semi")

n_teste, n_reais = df_teste.count(), df_reais.count()
# esperado: 200 removidas | 45.000 restantes
print(f"UCs de teste removidas: {n_br(n_teste)} | antes: {n_br(n_dedup)} | depois: {n_br(n_reais)}")
display(df_teste.select(cols_origem).orderBy("id_uc").limit(20))

## 7. Bairro com acentuação corrompida (log U6)

**O problema:** a busca da exploração encontrou 388 linhas com acentuação suspeita no bairro (32 delas são `Rio Âmbar`, que está correto). As demais têm **mojibake**: o texto foi gravado em UTF-8 e lido como Windows-1252, então `Jordão` virou `JordÃ£o` e `Brasília` virou `BrasÃ­lia`. Só a coluna `bairro` tem isso (exploração, seção 6).

**Decisão:** desfazer a troca de codificação (o texto volta para bytes em Windows-1252 e é lido de novo como UTF-8). Em alguns valores, um caractere invisível se perdeu no caminho (ex.: `Nossa Senhora do Ã`, `Ãgua Fria`) e a reversão automática não funciona. Esses ficam num **mapa manual**, conferido um a um.

**Cuidado:** `Rio Âmbar` (32 linhas) está **correto** (Âmbar é com acento circunflexo) e não pode ser alterado. A função só mexe em textos com `Ã`, então ele não é tocado.

**Como é aplicado:** há poucas centenas de bairros distintos, então a correção é calculada em Python para cada valor distinto e aplicada por `join` (rápido e determinístico).

**Conferência:** nenhum bairro com `Ã` depois da correção; a tabela mostra cada valor antes e depois.

In [0]:
# Tratamento 4 · mojibake em bairro: reverte a codificação; o que não reverte vai pelo mapa manual
def corrige_mojibake(s):
    """Desfaz texto UTF-8 lido como Windows-1252 (ex.: 'JordÃ£o' -> 'Jordão').
    Só age em textos com 'Ã'; se não conseguir reverter, devolve o texto original."""
    if s is None or "Ã" not in s:
        return s
    b = bytearray()
    for ch in s:
        try:
            b += ch.encode("cp1252")
        except UnicodeEncodeError:
            if ord(ch) < 256:          # caracteres de controle que o cp1252 não mapeia
                b.append(ord(ch))
            else:
                return s
    try:
        return b.decode("utf-8")
    except UnicodeDecodeError:
        return s

# Casos em que um caractere invisível se perdeu e a reversão automática não funciona
mapa_manual = {
    "Nossa Senhora do Ã": "Nossa Senhora do Ó",
    "Ãgua Fria": "Água Fria",
    "Caixa D'Ãgua": "Caixa D'Água",
    "Alto da Caixa D'Ãgua": "Alto da Caixa D'Água",
    "Ãguas Compridas": "Águas Compridas",
    "Ãrea Rural do Cabo de Santo Agostinho": "Área Rural do Cabo de Santo Agostinho",
    "Rio Ãmbar": "Rio Âmbar",
}

bairros = [r["bairro"] for r in df_reais.select("bairro").distinct().collect()]
pares = []
for original in bairros:
    limpo = None if original is None else original.strip()
    corrigido = mapa_manual.get(limpo, corrige_mojibake(limpo))
    pares.append((original, corrigido))
de_para = spark.createDataFrame(pares, "bairro string, bairro_corrigido string")

df_bairro = (df_reais.join(F.broadcast(de_para), "bairro", "left")
             .withColumn("bairro_original", F.col("bairro"))
             .withColumn("bairro", espacos(F.coalesce(F.col("bairro_corrigido"), F.col("bairro"))))
             .drop("bairro_corrigido"))

mudou = df_bairro.filter(F.col("bairro_original").contains("Ã"))
# esperado: 356 linhas corrigidas (388 da exploração menos as 32 de Rio Âmbar) | 0 bairros ainda com Ã
print("linhas com mojibake corrigidas:", n_br(mudou.count()))
print("bairros ainda com 'Ã':", n_br(df_bairro.filter(F.col("bairro").contains("Ã")).count()))
display(mudou.groupBy("bairro_original", "bairro").count().orderBy(F.desc("count")))
# Rio Âmbar é legítimo e não pode mudar (esperado: 32 + as 2 de 'Rio Ãmbar' corrigidas = 34)
print("Rio Âmbar depois da correção:", n_br(df_bairro.filter(F.col("bairro") == "Rio Âmbar").count()))

## 8. Categorias: `uf`, `municipio`, `classe`, `status`, `subgrupo_tarifario` (log U4 e U5)

**Decisão:** comparar as grafias pela forma padronizada (`sem_acento`) e mapear para um valor canônico.

| Coluna | Regra | Evidência |
|---|---|---|
| `uf` | `PE`, `P.E.` e `Pernambuco` (em qualquer grafia) → `PE`; pontos e espaços são ignorados na comparação | 6.990 linhas fora de `PE` (1.666 delas como `P.E.`); todas as UCs são de Pernambuco |
| `municipio` | grafias → nome oficial, com acento (14 municípios da Região Metropolitana do Recife) | o código do alimentador confirma o município (seção 11) |
| `classe` | grafias → `RESIDENCIAL`, `COMERCIAL`, `INDUSTRIAL`, `RURAL`, `PODER PUBLICO` | 5 classes do glossário do case |
| `status` | `upper(trim)` | só `ATIVA` |
| `subgrupo_tarifario` | `upper(trim)` | `B1`, `B2`, `B3`, `A4` |

Valor fora do mapa vira **nulo de propósito**: a checagem de não nulo falha e avisa que apareceu uma grafia nova. A célula mostra os valores não mapeados **antes** das checagens.

In [0]:
# Tratamento 5 · categorias para valores canônicos
def mapear(col_padronizada, mapa):
    expr = None
    for origem, destino in mapa.items():
        expr = (F.when(col_padronizada == origem, F.lit(destino)) if expr is None
                else expr.when(col_padronizada == origem, F.lit(destino)))
    return expr  # sem otherwise: fora do mapa vira nulo

mapa_uf = {"PE": "PE", "PERNAMBUCO": "PE"}

# chave = nome padronizado (sem acento, maiúsculo) -> nome oficial
municipios = {
    "ABREU E LIMA": "Abreu e Lima", "ARACOIABA": "Araçoiaba",
    "CABO DE SANTO AGOSTINHO": "Cabo de Santo Agostinho", "CAMARAGIBE": "Camaragibe",
    "IGARASSU": "Igarassu", "ILHA DE ITAMARACA": "Ilha de Itamaracá", "IPOJUCA": "Ipojuca",
    "ITAPISSUMA": "Itapissuma", "JABOATAO DOS GUARARAPES": "Jaboatão dos Guararapes",
    "MORENO": "Moreno", "OLINDA": "Olinda", "PAULISTA": "Paulista", "RECIFE": "Recife",
    "SAO LOURENCO DA MATA": "São Lourenço da Mata",
}
mapa_classe = {c: c for c in ["RESIDENCIAL", "COMERCIAL", "INDUSTRIAL", "RURAL", "PODER PUBLICO"]}

df_cat = (
    df_bairro
    .withColumn("uf_original", F.col("uf"))
    .withColumn("municipio_original", F.col("municipio"))
    .withColumn("classe_original", F.col("classe"))
    .withColumn("uf", mapear(F.regexp_replace(sem_acento(F.col("uf")), "[^A-Z]", ""), mapa_uf))  # 'P.E.' -> 'PE'
    .withColumn("municipio", mapear(sem_acento(F.col("municipio")), municipios))
    .withColumn("classe", mapear(F.regexp_replace(sem_acento(F.col("classe")), "_", " "), mapa_classe))
    .withColumn("status", F.upper(F.trim("status")))
    .withColumn("subgrupo_tarifario", F.upper(F.trim("subgrupo_tarifario")))
)

# Valores que NÃO entraram no mapa (esperado: nenhuma linha nas três tabelas)
for c in ["uf", "municipio", "classe"]:
    print(f"{c}: valores sem mapeamento")
    display(df_cat.filter(F.col(c).isNull()).groupBy(f"{c}_original").count())

# esperado: uf alterada em 6.990 linhas
print("uf alterada:", n_br(df_cat.filter(F.col("uf") != F.col("uf_original")).count()))
display(df_cat.groupBy("classe", "subgrupo_tarifario").count().orderBy("classe"))
display(df_cat.groupBy("municipio").count().orderBy(F.desc("count")))

## 9. Textos de endereço, `cep` e coordenadas (log U5 e U7)

**Decisão:**
- `logradouro`, `numero` e `alimentador`: `trim` e espaços internos simples (havia espaço no fim do logradouro).
- `cep`: manter só os dígitos quando o valor tem a forma de CEP (`99999-999` ou `99999999`); **qualquer outra forma vira nulo**. As 1.383 linhas inválidas estão em 7 formas que não são CEP (exploração, seção 5). Inventar um CEP seria pior que deixar vazio, e o endereço continua identificado pelo logradouro e pelo bairro.
- `latitude` e `longitude`: `double`.

**Conferência:** 1.383 CEPs anulados; nenhuma coordenada que deixe de virar número.

In [0]:
# Tratamento 6 · endereço, cep e coordenadas
cep_valido = F.trim("cep").rlike("^[0-9]{5}-?[0-9]{3}$")

df_end = (
    df_cat
    .withColumn("cep_original", F.col("cep"))
    .withColumn("cep", F.when(cep_valido, F.regexp_replace(F.trim("cep"), "-", "")))
    .withColumn("logradouro", espacos(F.col("logradouro")))
    .withColumn("numero", espacos(F.col("numero")))
    .withColumn("alimentador", F.upper(espacos(F.col("alimentador"))))
    .withColumn("latitude", F.expr("try_cast(trim(latitude) as double)"))
    .withColumn("longitude", F.expr("try_cast(trim(longitude) as double)"))
)

# esperado: 1.383 CEPs anulados (7 formas)
anulados = df_end.filter(F.col("cep_original").isNotNull() & F.col("cep").isNull())
print("CEPs anulados:", n_br(anulados.count()))
display(anulados.groupBy(F.regexp_replace(F.regexp_replace("cep_original", "[0-9]", "9"), "[A-Za-z]", "A").alias("forma"))
        .count().orderBy(F.desc("count")))
print("coordenadas que não viraram número:",
      n_br(df_end.filter(F.col("latitude").isNull() | F.col("longitude").isNull()).count()))

## 10. `data_ligacao` (log U9)

**Conversão:** cada padrão com o seu formato, a mesma convenção das outras tabelas (hífen = mês primeiro; ponto e barra = dia primeiro). Se a coluna só tiver ISO, os outros ramos simplesmente não são usados.

**O problema:** 90 UCs têm ligação **antes de 1950** (ex.: `UC105409` ligada em `1916-01-01`, com o primeiro medidor instalado em `2021-07-25`). Quase todas caem em `01/01`, o que indica valor de preenchimento, e não data real: só 1 das 90 tem o mesmo dia e mês da instalação do primeiro medidor.

**Gabarito:** em **todas** as outras 44.910 UCs, a data de ligação é **igual** à data de instalação do primeiro medidor (`silver.medidores`). Isso faz sentido: a UC é ligada quando o medidor é instalado.

**Decisão:** substituir as 90 datas pela instalação do 1º medidor e marcar `data_ligacao_estimada = true`. Assim a data fica coerente com o resto do cadastro, e quem usar a coluna sabe quais foram estimadas.
*Alternativa considerada:* deixar nulo com a mesma flag. Mais conservadora, mas tira 90 UCs de qualquer análise por tempo de ligação (por exemplo, "UCs novas por ano"), mesmo havendo uma estimativa com 100% de acerto nos outros casos.

**Conferência:** 90 estimadas; depois da correção, ligação = instalação do 1º medidor em 45.000 de 45.000.

In [0]:
# Tratamento 7 · data_ligacao: converte por padrão e estima as datas impossíveis
formatos = {
    "9999-99-99": "yyyy-MM-dd",
    "99-99-9999": "MM-dd-yyyy",
    "99.99.9999": "dd.MM.yyyy",
    "99/99/9999": "dd/MM/yyyy",
}
data_convertida = None
for padrao, fmt in formatos.items():
    cond = F.col("padrao_data") == padrao
    valor = F.expr(f"to_date(try_to_timestamp(trim(data_ligacao), '{fmt}'))")
    data_convertida = F.when(cond, valor) if data_convertida is None else data_convertida.when(cond, valor)

prim_med = (spark.table(f"{silver_schema}.medidores")
            .groupBy("id_uc").agg(F.min("data_instalacao").alias("instalacao_1o_medidor")))

df_lig = (
    df_end
    .withColumn("data_ligacao_original", F.col("data_ligacao"))
    .withColumn("padrao_data", F.regexp_replace(F.trim("data_ligacao"), "[0-9]", "9"))
    .withColumn("data_ligacao", data_convertida)
    .join(prim_med, "id_uc", "left")
    .withColumn("data_ligacao_estimada", F.col("data_ligacao") < F.lit("1950-01-01").cast("date"))
    .withColumn("data_ligacao", F.when(F.col("data_ligacao_estimada"), F.col("instalacao_1o_medidor"))
                                 .otherwise(F.col("data_ligacao")))
)

display(df_lig.groupBy("padrao_data").count().orderBy(F.desc("count")))
# esperado: 0 não convertidas | 90 estimadas | 45.000 iguais à instalação do 1º medidor
print("data_ligacao não convertida:", n_br(df_lig.filter("data_ligacao IS NULL").count()))
print("datas estimadas:", n_br(df_lig.filter("data_ligacao_estimada").count()))
print("ligação = instalação do 1º medidor:",
      n_br(df_lig.filter("data_ligacao = instalacao_1o_medidor").count()), "de", n_br(df_lig.count()))
display(df_lig.filter("data_ligacao_estimada")
        .select("id_uc", "data_ligacao_original", "instalacao_1o_medidor", "data_ligacao").limit(10))

## 11. Seleção final das colunas (log U10)

**Dados pessoais (`cpf_cnpj`, `nome_titular`):** ficam na Silver, que é uma camada interna. A Gold, o portal e o agente de IA só recebem versões **mascaradas** (regra no contrato da Gold), em linha com a LGPD.

Colunas auxiliares (`*_original`, `padrao_data`, `instalacao_1o_medidor`) ficam de fora. `source_file` é mantida para auditoria, como nas outras Silvers.

In [0]:
colunas_finais = [
    "id_uc", "nome_titular", "cpf_cnpj", "tipo_pessoa",
    "logradouro", "numero", "bairro", "municipio", "uf", "cep", "latitude", "longitude",
    "classe", "subgrupo_tarifario", "alimentador", "status",
    "data_ligacao", "data_ligacao_estimada",
    "source_file",
]
# confere se nenhuma coluna da Bronze ficou de fora sem querer
esquecidas = [c for c in cols_origem if c not in colunas_finais]
print("colunas da Bronze fora da Silver:", esquecidas)  # esperado: []

df_silver = df_lig.select(*colunas_finais)
df_silver.printSchema()

## 12. Checagens de qualidade

Rodam sobre o resultado, **antes** de gravar. Cada uma protege uma decisão:

| Checagem | Protege |
|---|---|
| Total = 45.000 e `id_uc` único | deduplicação (U2) e UCs de teste (U3) |
| Colunas-chave não nulas | mapas de categoria (U4, U5) e datas (U9) |
| `id_uc` no padrão; documento com 11 ou 14 dígitos e único | formatos (U8) |
| Domínios de `uf`, `classe`, `subgrupo`, `status`, `municipio` | mapas (U4, U5) |
| Classe × subgrupo conforme o glossário | erro de cadastro afetaria o **preço** no diagnóstico |
| Alimentador no padrão e com o código do município da UC | denominador do **DEC/FEC** por alimentador |
| `cep` nulo ou 8 dígitos; coordenadas na região | U7 e tipos numéricos |
| Bairro sem mojibake | U6 |
| Ligação entre 1950 e o fim da extração e igual à instalação do 1º medidor | U9 |
| Cada UC com exatamente 1 medidor ativo | coerência com `silver.medidores` (visão 360) |
| Todo `id_uc` de `silver.medidores` e de `silver.leituras` existe aqui | as outras Silvers não ficam sem cadastro |

A contagem de 45.000 é uma proteção desta extração: com dados novos, o valor esperado muda.

In [0]:
# Checagens de qualidade antes de gravar (condição = o que deve ser verdade em todas as linhas)
t = "silver.unidades_consumidoras"
dq_results.clear()  # se esta célula for reexecutada, não acumula resultados antigos

def ok(cond):
    """Condição nula conta como falha."""
    return F.coalesce(cond, F.lit(False))

# Volume e chave
dq_check_contagem(t, "total de linhas = 45.000", df_silver, 45_000)
dq_check_unique(t, "id_uc único", df_silver, ["id_uc"])

# Nulos
for coluna in ["id_uc", "nome_titular", "cpf_cnpj", "tipo_pessoa", "logradouro", "bairro", "municipio",
               "uf", "latitude", "longitude", "classe", "subgrupo_tarifario", "alimentador", "status",
               "data_ligacao", "data_ligacao_estimada"]:
    dq_check(t, f"{coluna} não nulo", df_silver, F.col(coluna).isNotNull())

# Formatos e domínios
dq_check(t, "id_uc no padrão UC + 6 dígitos", df_silver, ok(F.col("id_uc").rlike("^UC[0-9]{6}$")))
dq_check(t, "cpf_cnpj com 11 (PF) ou 14 (PJ) dígitos", df_silver, ok(F.col("cpf_cnpj").rlike("^([0-9]{11}|[0-9]{14})$")))
dq_check_unique(t, "cpf_cnpj único", df_silver, ["cpf_cnpj"])
dq_check(t, "uf = PE", df_silver, ok(F.col("uf") == "PE"))
dq_check(t, "municipio entre os 14 da RMR", df_silver, ok(F.col("municipio").isin(list(municipios.values()))))
dq_check(t, "classe entre as 5 do glossário", df_silver, ok(F.col("classe").isin(list(mapa_classe))))
dq_check(t, "subgrupo B1, B2, B3 ou A4", df_silver, ok(F.col("subgrupo_tarifario").isin("B1", "B2", "B3", "A4")))
dq_check(t, "status = ATIVA", df_silver, ok(F.col("status") == "ATIVA"))

subgrupo_esperado = (F.when(F.col("classe") == "RESIDENCIAL", "B1")
                      .when(F.col("classe") == "RURAL", "B2")
                      .when(F.col("classe").isin("COMERCIAL", "PODER PUBLICO"), "B3")
                      .when(F.col("classe") == "INDUSTRIAL", "A4"))
dq_check(t, "classe x subgrupo conforme o glossário", df_silver, ok(F.col("subgrupo_tarifario") == subgrupo_esperado))

cod_municipio = {
    "ABR": "Abreu e Lima", "ARA": "Araçoiaba", "CAB": "Cabo de Santo Agostinho", "CAM": "Camaragibe",
    "IGA": "Igarassu", "ILH": "Ilha de Itamaracá", "IPO": "Ipojuca", "ITA": "Itapissuma",
    "JAB": "Jaboatão dos Guararapes", "MOR": "Moreno", "OLI": "Olinda", "PAU": "Paulista",
    "REC": "Recife", "SAO": "São Lourenço da Mata",
}
dq_check(t, "alimentador no padrão ALM-XXX-000", df_silver, ok(F.col("alimentador").rlike("^ALM-[A-Z]{3}-[0-9]{3}$")))
mun_do_alim = mapear(F.substring("alimentador", 5, 3), cod_municipio)
dq_check(t, "código do alimentador = município da UC", df_silver, ok(F.col("municipio") == mun_do_alim))

dq_check(t, "cep nulo ou 8 dígitos", df_silver, F.col("cep").isNull() | ok(F.col("cep").rlike("^[0-9]{8}$")))
# Faixa da RMR com folga para o litoral: Ilha de Itamaracá e a orla de Olinda chegam a −34,78
dq_check(t, "latitude e longitude na região metropolitana do Recife", df_silver,
         ok(F.col("latitude").between(-8.6, -7.5) & F.col("longitude").between(-35.4, -34.7)))
dq_check(t, "bairro sem mojibake", df_silver, ~ok(F.col("bairro").rlike("Ã|[\u0080-\u009f]")))

# Datas
dq_check(t, "data_ligacao entre 1950-01-01 e 2026-06-30", df_silver,
         ok(F.col("data_ligacao").between(F.lit("1950-01-01").cast("date"), F.lit("2026-06-30").cast("date"))))
dq_check(t, "data_ligacao = instalação do 1º medidor", df_silver.join(prim_med, "id_uc", "left"),
         ok(F.col("data_ligacao") == F.col("instalacao_1o_medidor")))

# Coerência com medidores
med = spark.table(f"{silver_schema}.medidores")
ativos = med.filter("status = 'ATIVO'").groupBy("id_uc").agg(F.count("*").alias("qtd_ativos"))
dq_check(t, "cada UC com exatamente 1 medidor ativo", df_silver.join(ativos, "id_uc", "left"),
         ok(F.col("qtd_ativos") == 1))

ids_uc = df_silver.select("id_uc").withColumn("existe", F.lit(True))
dq_check(t, "todo id_uc de silver.medidores existe aqui",
         med.select("id_uc").distinct().join(ids_uc, "id_uc", "left"), ok(F.col("existe")))

# Integração com leituras (leituras não têm id_uc direto: passa por medidores)
if spark.catalog.tableExists(f"{silver_schema}.leituras"):
    uc_leit = (spark.table(f"{silver_schema}.leituras").select("id_medidor").distinct()
               .join(med.select("id_medidor", "id_uc"), "id_medidor").select("id_uc").distinct())
    dq_check(t, "toda UC de silver.leituras existe aqui",
             uc_leit.join(ids_uc, "id_uc", "left"), ok(F.col("existe")))
    dq_check(t, "toda UC tem leitura", df_silver.join(uc_leit.withColumn("tem_leitura", F.lit(True)), "id_uc", "left"),
             ok(F.col("tem_leitura")))
else:
    print("[AVISO] silver.leituras ainda não existe: checagens de leituras puladas")

## 13. Registro das checagens em `silver.dq_log`

Gravado **sempre**, mesmo com falha (modo `append`). A célula mostra só a execução atual.

In [0]:
df_dq_log = spark.createDataFrame(dq_results)
df_dq_log.write.format("delta").mode("append").option("mergeSchema", "true") \
    .saveAsTable(f"{silver_schema}.dq_log")

display(
    spark.table(f"{silver_schema}.dq_log")
    .filter(F.col("table_name") == t)
    .orderBy(F.col("checked_at").desc())
    .limit(len(dq_results))
)

## 14. Gravação da Silver, só se todas as checagens passarem

Se qualquer checagem falhar, a tabela **não é gravada**: o notebook lista as falhas e é interrompido com erro. Se `silver.unidades_consumidoras` já existia, ela permanece como estava.

In [0]:
falhas = [r for r in dq_results if not r.passed]

if falhas:
    print(f"{len(falhas)} checagem(ns) falharam. {tabela_silver} NÃO foi gravada:")
    for r in falhas:
        print(f"  - {r.check_name}: {n_br(r.failed_rows)} falha(s) em {n_br(r.total_rows)} linhas")
    raise Exception(f"{tabela_silver} não gravada: {len(falhas)} checagem(ns) de qualidade falharam (veja silver.dq_log)")

(
    df_silver.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_silver)
)
print(f"Todas as {len(dq_results)} checagens passaram. {tabela_silver} gravada: "
      f"{n_br(spark.table(tabela_silver).count())} linhas")

## 15. Bronze × Silver, com as mesmas UCs

Para cada tratamento, as mesmas `id_uc` são mostradas nas duas tabelas: o **texto original** da Bronze ao lado do valor tratado. Nas repetições, a Bronze mostra duas ou mais linhas e a Silver, uma. As UCs de teste aparecem só na Bronze. Os documentos são exibidos mascarados, para não expor dados pessoais no output.

In [0]:
silver_t = spark.table(tabela_silver)
colunas_ver = ["id_uc", "nome_titular", "cpf_cnpj", "bairro", "municipio", "uf", "cep", "classe", "data_ligacao"]

def mascara(df):
    return df.withColumn("cpf_cnpj", F.concat(F.lit("***"), F.substring(F.regexp_replace("cpf_cnpj", "[^0-9]", ""), 4, 3), F.lit("***")))

def comparar(titulo, ids_df, n=3):
    ids = [r["id_uc"] for r in ids_df.select("id_uc").limit(n).collect()]
    print(f"=== {titulo} | ids: {ids}")
    print("Bronze (texto original):")
    display(mascara(bronze.filter(F.col("id_uc").isin(ids)).select(*colunas_ver)).orderBy("id_uc"))
    print("Silver (tratado):")
    display(mascara(silver_t.filter(F.col("id_uc").isin(ids))).orderBy("id_uc"))

comparar("Repetições com grafias diferentes", df_sem_cab.groupBy("id_uc").count().filter("count > 1"), n=2)
comparar("UCs de teste (só na Bronze)", df_teste, n=2)
comparar("Bairro com mojibake", df_sem_cab.filter(F.col("bairro").contains("Ã")), n=3)
comparar("uf fora do padrão", df_sem_cab.filter(F.trim("uf") != "PE"), n=2)
comparar("cep inválido", df_sem_cab.filter(~F.trim("cep").rlike("^[0-9]{5}-?[0-9]{3}$")), n=3)
comparar("data_ligacao antes de 1950", silver_t.filter("data_ligacao_estimada"), n=3)